# 🔍 RAG com vários documentos

Este projeto implementa um sistema de RAG (Retrieval-Augmented Generation) utilizando:

- API da Groq
- Modelo llama-3.3-70b-versatile
- Vetorização com SentenceTransformers
- Banco vetorial FAISS
- Suporte para:
  - PDF
  - DOCX
  - XLSX
  - TXT


In [ ]:
# Instalação das Bibliotecas
!pip install -q groq sentence-transformers faiss-cpu pypdf python-docx pandas openpyxl unstructured nltk

In [ ]:
# Imports
import os
import faiss
import nltk
import numpy as np
import pandas as pd

from google.colab import files

from groq import Groq

from pypdf import PdfReader
from docx import Document

from sentence_transformers import SentenceTransformer

In [ ]:
# Download de Recursos do NLTK
nltk.download('punkt')

In [ ]:
# Configurar API KEY da Groq
GROQ_API_KEY = "SUA CHAVE AQUI"

In [ ]:
# Inicializar Modelo de Embeddings
embedding_model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

In [ ]:
# Upload dos Documentos
"""Você poderá enviar:
- PDFs
- DOCX
- XLSX
- TXT
"""

uploaded = files.upload()

In [ ]:
# Funções para Leitura de Arquivos
def read_pdf(file_path):
    text = ""

    reader = PdfReader(file_path)

    for page in reader.pages:
        content = page.extract_text()

        if content:
            text += content + "\n"

    return text


def read_docx(file_path):
    doc = Document(file_path)

    text = "\n".join(
        [paragraph.text for paragraph in doc.paragraphs]
    )

    return text


def read_xlsx(file_path):
    xls = pd.ExcelFile(file_path)

    text = ""

    for sheet in xls.sheet_names:
        df = pd.read_excel(file_path, sheet_name=sheet)

        text += f"\nPlanilha: {sheet}\n"

        text += df.astype(str).to_string()

    return text


def read_txt(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()

In [ ]:
# Função Geral de Extração
def extract_text(file_path):
    extension = file_path.split('.')[-1].lower()

    if extension == 'pdf':
        return read_pdf(file_path)

    elif extension == 'docx':
        return read_docx(file_path)

    elif extension == 'xlsx':
        return read_xlsx(file_path)

    elif extension == 'txt':
        return read_txt(file_path)

    else:
        return ""

In [ ]:
# Ler Todos os Arquivos
documents = []

for file_name in uploaded.keys():

    text = extract_text(file_name)

    documents.append({
        "file_name": file_name,
        "content": text
    })

print(f"{len(documents)} documentos carregados.")

In [ ]:
# Função de Chunking
def chunk_text(text, chunk_size=500):

    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):

        chunk = " ".join(words[i:i + chunk_size])

        chunks.append(chunk)

    return chunks

In [ ]:
# Criar Chunks
all_chunks = []

metadata = []

for doc in documents:

    chunks = chunk_text(doc["content"])

    for chunk in chunks:

        all_chunks.append(chunk)

        metadata.append({
            "source": doc["file_name"]
        })

print(f"{len(all_chunks)} chunks criados.")

In [ ]:
# Gerar Embeddings
embeddings = embedding_model.encode(all_chunks)

embeddings = np.array(embeddings).astype('float32')

In [ ]:
# Criar Banco Vetorial FAISS
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS indexado.")

In [ ]:
# Função de Busca Semântica
def search(query, top_k=3):

    query_embedding = embedding_model.encode([query])

    query_embedding = np.array(query_embedding).astype('float32')

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx in indices[0]:

        results.append({
            "chunk": all_chunks[idx],
            "metadata": metadata[idx]
        })

    return results

In [ ]:
# Inicializar Cliente Groq
client = Groq(
    api_key=GROQ_API_KEY
)

In [ ]:
# Função RAG
def ask_rag(question):

    results = search(question)

    context = ""

    for result in results:

        source = result["metadata"]["source"]

        chunk = result["chunk"]

        context += f"\nFonte: {source}\n"

        context += chunk + "\n"

    prompt = f"""
    Você é um assistente especialista em análise documental.

    Responda usando exclusivamente o contexto abaixo.

    CONTEXTO:
    {context}

    PERGUNTA:
    {question}
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=1024
    )

    return response.choices[0].message.content

In [ ]:
# Fazer Perguntas ao RAG
while True:

    question = input("\nPergunta: ")

    if question.lower() == "sair":
        break

    answer = ask_rag(question)

    print("\nResposta:\n")
    print(answer)